# Colliding-Message Handling in the Kerchunk Reference Generator

Demonstrates how the generator keeps every message when several GRIB2 messages
share the same parameter, level, and lead time, instead of silently overwriting
the earlier message's chunk references.

- **Probability products** that differ only in their threshold become separate
  variables named for the threshold, with `p` for a decimal point, e.g.
  `PTYPE_prob_between_1_and_2` or `CEIL_prob_below_304p8`. The limits are also
  written to each variable's `.zattrs` (`thresholdLowerLimit`,
  `thresholdUpperLimit`).
- **Messages with fully identical identifying metadata** (e.g. the smartinit
  surface TMP pair in #259) are emitted as `_dup` variables with a warning.

The example uses a public NBM core file from NOMADS. NBM `PTYPE` output
carries four surface probability messages that differ only in their category
bounds.

Requires this PR's branch installed (`pip install -e .`) plus `xarray`,
`fsspec`, and `zarr`.

In [ ]:
import datetime
import urllib.request
from pathlib import Path

# NOMADS retains roughly the last two days of NBM output; default to yesterday 00z.
date = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)).strftime("%Y%m%d")
url = f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/blend/prod/blend.{date}/00/core/blend.t00z.core.f001.ak.grib2"

grib_file = Path("blend.t00z.core.f001.ak.grib2")
if not grib_file.exists():
    print("downloading", url)
    urllib.request.urlretrieve(url, grib_file)
print(grib_file, f"{grib_file.stat().st_size / 1e6:.1f} MB")

## Generate a manifest filtered to PTYPE

The filter keeps the example focused; the same handling applies to an
unfiltered manifest.

In [ ]:
import warnings
from grib2io.kerchunk import ReferenceGenerator

gen = ReferenceGenerator(str(grib_file), filters={"shortName": "PTYPE"})
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    gen.to_json("blend_ptype.json")

for w in caught:
    print("WARNING:", w.message)

No collision warnings are expected: each PTYPE message is named for its
probability bounds.

In [ ]:
import json

refs = json.load(open("blend_ptype.json"))["refs"]
variables = sorted({k.split("/")[0] for k in refs if k.endswith("/.zarray") and k.count("/") == 1})
print(variables)

## Open the manifest and read every variant

All four messages remain reachable. On `main`, only one of these would exist
in the manifest and the other three messages' bytes would not be referenced
anywhere.

In [ ]:
import fsspec
import numpy as np
import xarray as xr
import grib2io.codecs  # registers the codec

fs = fsspec.filesystem("reference", fo="blend_ptype.json", skip_instance_cache=True)
ds = xr.open_dataset(fs.get_mapper(""), engine="zarr", backend_kwargs={"consolidated": False})

for name in variables:
    if not name.startswith("PTYPE"):
        continue
    vals = np.asarray(ds[name].values).ravel()
    finite = vals[np.isfinite(vals)]
    attrs = ds[name].attrs
    print(
        f"{name:28s} limits=[{attrs.get('thresholdLowerLimit')}, {attrs.get('thresholdUpperLimit')})"
        f"  finite {finite.size:,}/{vals.size:,}  max {finite.max():.0f}%"
    )

Each variable holds a different probability field, so keeping only one of
them would discard real, distinct data.

## When `_dup` is still used

`_dup` is reserved for messages whose identifying metadata is fully identical,
so nothing can be derived to tell them apart. The generator cannot know which
one a consumer intends; raising would fail whole files over one oddly-encoded
product, and skipping or overwriting is silent data loss. Emitting every such
message with a warning preserves the file's full contents and leaves the
choice with the consumer.